In [0]:
%sql
create catalog if not exists weather_catalog;

create schema if not exists weather_catalog.data;
create schema if not exists weather_catalog.bronze;

create volume if not exists weather_catalog.data.raw_files;
create volume if not exists weather_catalog.data.reference_file;

create volume if not exists weather_catalog.data.schema;
create volume if not exists weather_catalog.bronze.checkpoint;

In [0]:
import pyspark.sql.functions as f

table_name = "weather_catalog.bronze.bronze_weather_tbl"

df_raw = spark.readStream \
.format("cloudFiles") \
.option("cloudFiles.format", "csv") \
.option("header", "true") \
.option("quote", '"') \     # treats entire JSON string as one field
.option("escape", '"') \   # converts "" → " inside the JSON string
.option("multiLine", "true") \
.option("cloudFiles.schemaLocation", "/Volumes/weather_catalog/data/schema") \
.option("cloudFiles.schemaEvolutionMode", "rescue") \
.option("cloudFiles.inferColumnTypes", "true") \
.option("maxFilesPerTrigger", 1) \
.load("/Volumes/weather_catalog/data/raw_files/")

In [0]:
# ── foreachBatch — audit cols + write only ────────────────────────────────────
def write_bronze(batch_df, batch_id):

    if batch_df.count() == 0:
        print(f"Batch {batch_id}: empty, skipping...")
        return

    batch_df = batch_df \
        .withColumn("ingestion_time", f.current_timestamp()) \
        .withColumn("source_file",    f.col("_metadata.file_path")) \
        .withColumn("batch_id",       f.col("_metadata.file_modification_time").cast("string")) \
        .withColumn("load_date",      f.current_date()) \
        .withColumn("record_status",  f.lit("valid"))

    if not spark.catalog.tableExists(table_name):
        batch_df.write \
            .format("delta") \
            .mode("append") \
            .partitionBy("load_date") \
            .option("mergeSchema", "true") \
            .option("delta.enableChangeDataFeed", "true") \
            .saveAsTable(table_name)

    else:
        batch_df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(table_name)

df_raw.writeStream \
    .foreachBatch(write_bronze) \
    .option("checkpointLocation", "/Volumes/weather_catalog/bronze/checkpoint") \
    .trigger(availableNow=True) \
    .start() \
    .awaitTermination()

print("Bronze stream done ")

Bronze stream done 


In [0]:
%sql
select * from weather_catalog.bronze.bronze_weather_tbl;

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,rainfall,station_id,data_provider,_rescued_data,ingestion_time,source_file,batch_id,load_date,record_status
Chennai,India,2026-02-22T00:00:00.000Z,37.8,74,11.4,1009.7,Haze,1.0,STN_CHN_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Bangalore,India,2026-02-22T00:00:00.000Z,27.5,57,11.9,913.3,Foggy,1.5,STN_BLR_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Hyderabad,India,2026-02-22T00:00:00.000Z,39.0,69,21.4,999.3,Clear,6.0,STN_HYD_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Mumbai,India,2026-02-22T00:00:00.000Z,35.5,83,22.6,1012.9,Rainy,5.2,STN_MUM_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Delhi,India,2026-02-22T00:00:00.000Z,25.0,46,28.8,997.3,Foggy,7.3,STN_DEL_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Pune,India,2026-02-22T00:00:00.000Z,31.0,69,18.2,940.8,Clear,5.7,STN_PNE_01,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Chennai,India,2026-02-22T01:00:00.000Z,31.0,74,16.4,1009.5,Haze,10.7,STN_CHN_01,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Bangalore,India,2026-02-22T01:00:00.000Z,31.1,64,5.9,913.8,Partly Cloudy,3.2,STN_BLR_01,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Hyderabad,India,2026-02-22T01:00:00.000Z,31.6,64,13.0,999.5,Cloudy,6.8,STN_HYD_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid
Mumbai,India,2026-02-22T01:00:00.000Z,32.2,87,17.4,1008.4,Foggy,13.2,STN_MUM_02,WeatherPro,null,2026-04-14T05:27:27.315Z,/Volumes/weather_catalog/data/raw_files/weather_10.csv,2026-04-14 05:26:47,2026-04-14,valid


In [0]:
# df = spark.read.table("weather_catalog.bronze.bronze_weather_tbl")
# df.display()

In [0]:
# spark.sql("describe history weather_catalog.bronze.bronze_weather_tbl").display()

In [0]:
# dbutils.fs.rm("/Volumes/weather_catalog/bronze/checkpoint", recurse=True)
# dbutils.fs.rm("/Volumes/weather_catalog/data/schema", recurse=True)
# spark.sql("drop table weather_catalog.bronze.bronze_weather_tbl")

In [0]:
# df_raw = spark.readStream \
# .format("cloudFiles")\
# .option("cloudFiles.format", "csv")\
# .option("header", "true")\
# .option("quote", '"') \
# .option("escape", '"') \
# .option("multiLine", "true") \
# .option("cloudFiles.schemaLocation", "/Volumes/weather_catalog/data/schema")\
# .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
# .option("cloudFiles.inferColumnTypes", "true")\
# .option("rescuedDataColumn", "_rescued_data")\
# .option("maxFilesPerTrigger", 1)\
# .load("/Volumes/weather_catalog/data/temp_vol/")


# import pyspark.sql.functions as f
# import pyspark.sql.types as t
# known_keys = {
#     "temperature"           : "double",
#     "wind_speed"            : "double",
#     "pressure"              : "double",
#     "rainfall"              : "double",
#     "uv_index"              : "double",
#     "visibility"            : "double",
#     "feels_like_temperature": "double",
#     "humidity"              : "integer",
#     "weather_condition"     : "string",
#     "condition"             : "string"
# }

# if "sensor_payload" in df_raw.columns:
#     df_parsed = df_raw.withColumn(
#         "json_map",
#         f.from_json(
#             f.col("sensor_payload"),
#             t.MapType(t.StringType(), t.StringType())
#         )
#     )
# else:
#     df_parsed = df_raw.withColumn("json_map", f.lit(None))

# df_transformed = df_parsed

# for key, cast_type in known_keys.items():
#     df_transformed = df_transformed.withColumn(
#         key,
#         f.coalesce(
#             f.col(key).cast(cast_type),
#             f.col("json_map")[key].cast(cast_type)
#         )
#     )
# def write_with_cdf(batch_df, batch_id):
    
#     table_name = "weather_catalog.bronze.bronze_weather_raw"

#     table_exists = spark.catalog.tableExists(table_name)

#     batch_df = batch_df.withColumn("ingestion_time", f.current_timestamp()) \
#                     .withColumn("source_file", f.col("_metadata.file_path")) \
#                     .withColumn("batch_id", f.col('_metadata.file_modification_time').cast("string")) \
#                     .withColumn('load_date', f.current_date()) \
#                     .withColumn('record_status', f.lit('valid'))

#     if "event_time" in batch_df.columns:
#         batch_df = batch_df.withColumn("event_time", f.col("event_time").cast("timestamp"))
    
#     if "temperature" in batch_df.columns:
#         batch_df = batch_df.withColumn("temperature", f.col("temperature").cast("double"))
    
#     if "humidity" in batch_df.columns:
#         batch_df = batch_df.withColumn("humidity", f.col("humidity").cast("int"))
    
#     if "wind_speed" in batch_df.columns:
#         batch_df = batch_df.withColumn("wind_speed", f.col("wind_speed").cast("double"))
    
#     if "pressure" in batch_df.columns:
#         batch_df = batch_df.withColumn("pressure", f.col("pressure").cast("double"))
    
#     if "rainfall" in batch_df.columns:
#         batch_df = batch_df.withColumn("rainfall", f.col("rainfall").cast("double"))
    
#     if "uv_index" in batch_df.columns:
#         batch_df = batch_df.withColumn("uv_index", f.col("uv_index").cast("double"))
    
#     if "visibility" in batch_df.columns:
#         batch_df = batch_df.withColumn("visibility", f.col("visibility").cast("double"))

#     if "condition" in batch_df.columns and "weather_condition" not in batch_df.columns:
#         batch_df = batch_df.withColumnRenamed("condition", "weather_condition")

#     if not table_exists:
#         batch_df.write \
#             .format("delta") \
#             .mode("append") \
#             .partitionBy("load_date") \
#             .option("delta.enableChangeDataFeed", "true") \
#             .option("mergeSchema", "true") \
#             .saveAsTable(table_name)
#     else:
#         print(f"Batch {batch_id}: Appending to existing table...")
#         batch_df.write \
#             .format("delta") \
#             .mode("append") \
#             .option("mergeSchema", "true") \
#             .saveAsTable(table_name)

# df_transformed.writeStream \
#     .format("delta") \
#     .option("checkpointLocation", "/Volumes/weather_catalog/bronze/checkpoint") \
#     .option("mergeSchema", "true") \
#     .outputMode("append") \
#     .trigger(availableNow=True) \
#     .foreachBatch(write_with_cdf) \
#     .start() \
#     .awaitTermination()

In [0]:
# df = spark.read.table("weather_catalog.bronze.bronze_weather_raw")
# df.display()

In [0]:
# from delta.tables import DeltaTable

# existing = DeltaTable.forName(spark, "weather_catalog.bronze.bronze_weather_raw")

# print("=== Existing Table Schema ===")
# existing.toDF().printSchema()

# print("=== Incoming DataFrame Schema ===")
# df_raw.printSchema()

In [0]:
# import pyspark.sql.functions as f
# import pyspark.sql.types as t


# def write_with_cdf(batch_df, batch_id):
    
#     table_name = "weather_catalog.bronze.bronze_weather_raw"

#     table_exists = spark.catalog.tableExists(table_name)

#     batch_df = batch_df.withColumn("ingestion_time", f.current_timestamp()) \
#                     .withColumn("source_file", f.col("_metadata.file_path")) \
#                     .withColumn("batch_id", f.col('_metadata.file_modification_time').cast("string")) \
#                     .withColumn('load_date', f.current_date()) \
#                     .withColumn('record_status', f.lit('valid'))
#     if "condition" in batch_df.columns:
#         batch_df = batch_df.withColumnRenamed("condition", "weather_condition")

#     if not table_exists:
#         print(f"Batch {batch_id}: Creating table with CDF enabled...")
        
#         batch_df.write \
#             .format("delta") \
#             .mode("append") \
#             .partitionBy("load_date") \
#             .option("delta.enableChangeDataFeed", "true") \
#             .option("mergeSchema", "true") \
#             .saveAsTable(table_name)
#     else:
#         print(f"Batch {batch_id}: Appending to existing table...")
#         batch_df.write \
#             .format("delta") \
#             .mode("append") \
#             .option("mergeSchema", "true") \
#             .saveAsTable(table_name)

# df_raw.writeStream \
#     .format("delta") \
#     .option("checkpointLocation", "/Volumes/weather_catalog/bronze/checkpoint") \
#     .option("mergeSchema", "true") \
#     .outputMode("append") \
#     .trigger(availableNow=True) \
#     .foreachBatch(write_with_cdf) \
#     .start() \
#     .awaitTermination()

In [0]:
# spark.sql("""
#     SHOW TBLPROPERTIES weather_catalog.bronze.bronze_weather_raw
# """).filter("key = 'delta.enableChangeDataFeed'").show()

In [0]:
%sql
-- DESCRIBE history weather_catalog.bronze.bronze_weather_raw;

In [0]:
# # Read as plain text first — no parsing at all
# df_text = spark.read \
#     .option("header", "true") \
#     .text("/Volumes/weather_catalog/data/raw_files/weather_13.csv")

# df_text.show(3, truncate=False)

In [0]:
# df_batch = spark.read \
#     .format("csv") \
#     .option("header", "true") \
#     .option("quote", '"') \
#     .option("escape", '"') \
#     .option("multiLine", "true") \
#     .load("/Volumes/weather_catalog/data/raw_files/weather_13.csv")

# # See raw sensor_payload value
# df_batch.select("city", "sensor_payload").show(3, truncate=False)
# df_batch.display()


In [0]:
# # See the EXACT string stored in sensor_payload
# raw_value = df_batch.select("sensor_payload").limit(1).collect()[0][0]

# print(type(raw_value))
# print(repr(raw_value))   # repr shows escape chars, quotes exactly as stored
# print("---")
# print(raw_value)         # normal print

In [0]:
# import pyspark.sql.functions as f
# import pyspark.sql.types as t

# # ── Step 1: Parse JSON as Map ─────────────────────────────────────────────────
# df_parsed = df_raw.withColumn(
#     "json_map",
#     f.from_json(
#         f.col("sensor_payload"),
#         t.MapType(t.StringType(), t.StringType())
#     )
# )

# # ── Step 2: Discover keys — NO RDD, use collect() on DataFrame directly ───────
# json_keys = (
#     df_parsed
#     .select(f.explode(f.map_keys(f.col("json_map"))).alias("key"))
#     .distinct()
#     .orderBy("key")
#     .collect()                        # ✅ collect() on DataFrame — allowed on serverless
# )

# json_keys = [row["key"] for row in json_keys]   # ✅ plain Python list comprehension
# print("Keys found:", json_keys)

# # ── Step 3: Dynamically add columns ──────────────────────────────────────────
# numeric_keys = {"temperature", "wind_speed", "pressure",
#                  "uv_index", "visibility", "feels_like_temperature"}
# integer_keys = {"humidity"}

# df_final = df_parsed
# for key in json_keys:
#     if key in integer_keys:
#         df_final = df_final.withColumn(key, f.col("json_map")[key].cast("integer"))
#     elif key in numeric_keys:
#         df_final = df_final.withColumn(key, f.col("json_map")[key].cast("double"))
#     else:
#         df_final = df_final.withColumn(key, f.col("json_map")[key])  # string

# df_final = df_final.drop("json_map")

# df_final.printSchema()
# # df_final.display()

In [0]:
# import pyspark.sql.functions as f
# import pyspark.sql.types as t

# # ── Schema matching EXACT keys from your JSON ────────────────────────────────
# sensor_schema = t.StructType([
#     t.StructField("temperature",       t.DoubleType()),
#     t.StructField("humidity",          t.IntegerType()),
#     t.StructField("wind_speed",        t.DoubleType()),
#     t.StructField("pressure",          t.DoubleType()),
#     t.StructField("weather_condition", t.StringType()),
#     t.StructField("rainfall",          t.DoubleType()),
#     t.StructField("uv_index",          t.DoubleType()),
#     t.StructField("visibility",        t.DoubleType())
# ])

# # ── Parse directly — no cleaning needed, JSON is already clean ───────────────
# df_parsed = df_batch.withColumn(
#     "sensor_struct",
#     f.from_json(f.col("sensor_payload"), sensor_schema)
# )

# # ── Flatten struct into main dataframe ───────────────────────────────────────
# df_final = df_parsed \
#     .select("*", "sensor_struct.*") \
#     .drop("sensor_struct")

# # ── Verify ───────────────────────────────────────────────────────────────────
# df_final.select(
#     "city", "temperature", "humidity",
#     "wind_speed", "weather_condition", "visibility"
# ).show(5, truncate=False)

# df_final.printSchema()

In [0]:
# import pyspark.sql.functions as f
# import pyspark.sql.types as t

# table_name = "weather_catalog.bronze.bronze_weather_raw"

# known_keys = {
#     "temperature"           : "double",
#     "wind_speed"            : "double",
#     "pressure"              : "double",
#     "rainfall"              : "double",
#     "uv_index"              : "double",
#     "visibility"            : "double",
#     "feels_like_temperature": "double",
#     "humidity"              : "integer",
#     "weather_condition"     : "string",
#     "condition"             : "string"
# }

# # ── ReadStream — key fix is rescuedDataColumn ─────────────────────────────────
# df_raw = spark.readStream \
#     .format("cloudFiles") \
#     .option("cloudFiles.format", "csv") \
#     .option("header", "true") \
#     .option("quote", '"') \
#     .option("escape", '"') \
#     .option("multiLine", "true") \
#     .option("cloudFiles.schemaLocation", "/Volumes/weather_catalog/data/schema") \
#     .option("cloudFiles.inferColumnTypes", "true").option("cloudFiles.rescuedDataColumn", "_rescued_data") \
#     .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
#     .option("maxFilesPerTrigger", 1) \
#     .load("/Volumes/weather_catalog/data/temp_vol/")

# # ── Parse sensor_payload JSON as MapType ──────────────────────────────────────
# df_parsed = df_raw.withColumn(
#     "json_map",
#     f.from_json(
#         f.col("sensor_payload"),
#         t.MapType(t.StringType(), t.StringType())
#     )
# )

# # ── Dynamically extract known keys ────────────────────────────────────────────
# df_transformed = df_parsed
# for key, cast_type in known_keys.items():
#     df_transformed = df_transformed.withColumn(
#         key,
#         f.when(
#             f.col("json_map").isNotNull(),
#             f.col("json_map")[key].cast(cast_type)
#         ).otherwise(f.lit(None).cast(cast_type))
#     )

# # ── Audit columns + drop helpers ──────────────────────────────────────────────
# df_final = df_transformed \
#     .drop("json_map", "sensor_payload", "_rescued_data") \
#     .withColumn("ingestion_time", f.current_timestamp()) \
#     .withColumn("source_file",    f.col("_metadata.file_path")) \
#     .withColumn("batch_id",       f.col("_metadata.file_modification_time").cast("string")) \
#     .withColumn("load_date",      f.current_date()) \
#     .withColumn("record_status",  f.lit("valid"))

# # ── foreachBatch ──────────────────────────────────────────────────────────────
# def write_with_cdf(batch_df, batch_id):

#     if batch_df.count() == 0:
#         print(f"Batch {batch_id}: empty, skipping...")
#         return

#     if not spark.catalog.tableExists(table_name):
#         print(f"Batch {batch_id}: creating table with CDF...")

#         batch_df.write \
#             .format("delta") \
#             .mode("append") \
#             .partitionBy("load_date") \
#             .option("mergeSchema", "true") \
#             .option("delta.enableChangeDataFeed", "true") \
#             .saveAsTable(table_name)

#         print(f"Batch {batch_id}: table created + CDF enabled ✅")

#     else:
#         print(f"Batch {batch_id}: appending...")

#         batch_df.write \
#             .format("delta") \
#             .mode("append") \
#             .option("mergeSchema", "true") \
#             .saveAsTable(table_name)

#         print(f"Batch {batch_id}: append done ✅")

# # ── WriteStream ───────────────────────────────────────────────────────────────
# query = df_final.writeStream \
#     .foreachBatch(write_with_cdf) \
#     .option("checkpointLocation", "/Volumes/weather_catalog/bronze/checkpoint") \
#     .trigger(availableNow=True) \
#     .start()

# query.awaitTermination()
# print("Stream done ✅")

In [0]:
# df = spark.read.table("weather_catalog.bronze.bronze_weather_raw")
# df.display()